# NoShowGuard Capstone
## Part 1: Python & Data Foundations
### Task 1.1 — Read the file the foundations way

In [1]:

import os

print("appointments.csv exists:", os.path.exists("/content/appointments.csv"))


appointments.csv exists: True


In [2]:

import csv
import json

# Read the original appointments file
with open("appointments.csv", newline="") as file:
    reader = csv.DictReader(file)
    header = reader.fieldnames
    rows = list(reader)

print("Header:", header)
print("Total rows read:", len(rows))


Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows read: 1020


In [3]:

# Task 1.1 — Convert required columns to numeric types

def parse_row(row):
    converted_row = row.copy()

    converted_row["age"] = int(row["age"])
    converted_row["fee_inr"] = int(row["fee_inr"])
    converted_row["prior_no_shows"] = int(row["prior_no_shows"])
    converted_row["distance_km"] = float(row["distance_km"])

    return converted_row


# Test the function on the first row
parsed_row = parse_row(rows[0])

print("Age:", parsed_row["age"], type(parsed_row["age"]))
print("Fee:", parsed_row["fee_inr"], type(parsed_row["fee_inr"]))
print("Prior no-shows:", parsed_row["prior_no_shows"], type(parsed_row["prior_no_shows"]))
print("Distance:", parsed_row["distance_km"], type(parsed_row["distance_km"]))


Age: 60 <class 'int'>
Fee: 850 <class 'int'>
Prior no-shows: 1 <class 'int'>
Distance: 18.0 <class 'float'>


In [4]:

# Task 1.1 — Separate accepted and rejected records

accepted = []
rejected = []

for row in rows:
    try:
        converted_row = parse_row(row)
        accepted.append(converted_row)

    except ValueError:
        rejected.append(row)

print("Total rows:", len(rows))
print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))


Total rows: 1020
Accepted rows: 988
Rejected rows: 32


In [5]:

# Task 1.1 — Export rejected records to JSON

import json

with open("rejected_rows.json", "w") as file:
    json.dump(rejected, file, indent=2)

print("File created: rejected_rows.json")
print("Total rejected records saved:", len(rejected))


File created: rejected_rows.json
Total rejected records saved: 32


### Task 1.2 — Inspect and clean, pass 1

In [6]:

# Task 1.2 — Load and inspect the original dataset

import pandas as pd

df = pd.read_csv("appointments.csv")

print("Dataset shape:")
print(df.shape)

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nExact duplicate rows:")
print(df.duplicated().sum())


Dataset shape:
(1020, 14)

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values:
appointment_id      0
patie

In [7]:

# Task 1.2(a) — Remove exact duplicate rows

print("Rows before removing duplicates:", len(df))

df = df.drop_duplicates().copy()

print("Rows after removing duplicates:", len(df))
print("Remaining duplicate rows:", df.duplicated().sum())


Rows before removing duplicates: 1020
Rows after removing duplicates: 1000
Remaining duplicate rows: 0


In [8]:

# Task 1.2(b) — Standardize booking_channel

print("Unique booking channels before cleaning:")
print(df["booking_channel"].nunique())

df["booking_channel"] = (
    df["booking_channel"]
    .str.strip()
    .str.title()
)

print("\nUnique booking channels after cleaning:")
print(df["booking_channel"].nunique())

print("\nCleaned booking channel values:")
print(df["booking_channel"].unique())

print("\nRows after standardization:", len(df))


Unique booking channels before cleaning:
16

Unique booking channels after cleaning:
3

Cleaned booking channel values:
['Phone' 'Walk-In' 'App']

Rows after standardization: 1000


In [9]:

# Task 1.2(c) — Convert date columns

df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce"
)

df["booked_on"] = pd.to_datetime(
    df["booked_on"]
)

print("Date column data types:")
print(df[["appt_date", "booked_on"]].dtypes)

print("\nMissing appointment dates after conversion:")
print(df["appt_date"].isnull().sum())

print("\nRows after date conversion:", len(df))


Date column data types:
appt_date    datetime64[ns]
booked_on    datetime64[ns]
dtype: object

Missing appointment dates after conversion:
6

Rows after date conversion: 1000


In [10]:

# Task 1.2(d) — Drop rows with missing appointment date or status

print("Rows before removing missing records:", len(df))

print("Missing appointment dates:", df["appt_date"].isnull().sum())
print("Missing statuses:", df["status"].isnull().sum())

df = df.dropna(subset=["appt_date", "status"]).copy()

print("\nRows after removing missing records:", len(df))

print("Remaining missing appointment dates:",
      df["appt_date"].isnull().sum())

print("Remaining missing statuses:",
      df["status"].isnull().sum())


Rows before removing missing records: 1000
Missing appointment dates: 6
Missing statuses: 4

Rows after removing missing records: 990
Remaining missing appointment dates: 0
Remaining missing statuses: 0


### Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [11]:

# Task 1.3(a) — Identify and replace invalid ages

print("Missing ages before validation:",
      df["age"].isnull().sum())

invalid_age = (df["age"] < 1) | (df["age"] > 100)

print("Invalid ages found:", invalid_age.sum())

df.loc[invalid_age, "age"] = float("nan")

print("Missing ages after validation:",
      df["age"].isnull().sum())

print("Rows remaining:", len(df))


Missing ages before validation: 12
Invalid ages found: 5
Missing ages after validation: 17
Rows remaining: 990


In [12]:

# Task 1.3(b) — Fill missing values using medians

# Calculate medians before filling missing values
median_age = df["age"].median()
median_distance = df["distance_km"].median()

print("Median age:", median_age)
print("Median distance:", median_distance)

# Fill missing ages, round and convert to integer
df["age"] = (
    df["age"]
    .fillna(median_age)
    .round()
    .astype(int)
)

# Fill missing distances
df["distance_km"] = df["distance_km"].fillna(
    median_distance
)

print("\nMissing ages after filling:",
      df["age"].isnull().sum())

print("Missing distances after filling:",
      df["distance_km"].isnull().sum())

print("Rows remaining:", len(df))


Median age: 48.0
Median distance: 8.25

Missing ages after filling: 0
Missing distances after filling: 0
Rows remaining: 990


In [13]:

# Task 1.3(c) — IQR-based outlier capping

import numpy as np

for column in ["distance_km", "fee_inr"]:

    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)

    iqr = q3 - q1

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    # Count values outside the fences before capping
    capped_count = (
        (df[column] < lower_fence) |
        (df[column] > upper_fence)
    ).sum()

    # Cap outliers at the IQR fences
    df[column] = np.clip(
        df[column],
        lower_fence,
        upper_fence
    )

    print(f"\nColumn: {column}")
    print("Q1:", q1)
    print("Q3:", q3)
    print("IQR:", iqr)
    print("Lower fence:", lower_fence)
    print("Upper fence:", upper_fence)
    print("Values capped:", capped_count)



Column: distance_km
Q1: 4.625
Q3: 13.0
IQR: 8.375
Lower fence: -7.9375
Upper fence: 25.5625
Values capped: 26

Column: fee_inr
Q1: 500.0
Q3: 787.5
IQR: 287.5
Lower fence: 68.75
Upper fence: 1218.75
Values capped: 3


In [14]:

# Task 1.3(d) — Final rounding after outlier capping

df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = (
    df["fee_inr"]
    .round()
    .astype(int)
)

print("Maximum fee after rounding:", df["fee_inr"].max())

print("\nColumn data types:")
print(df[["age", "distance_km", "fee_inr"]].dtypes)

print("\nMissing values:")
print(df[["age", "distance_km", "fee_inr"]].isnull().sum())

print("\nRows remaining:", len(df))


Maximum fee after rounding: 1219

Column data types:
age              int64
distance_km    float64
fee_inr          int64
dtype: object

Missing values:
age            0
distance_km    0
fee_inr        0
dtype: int64

Rows remaining: 990


In [15]:

# Task 1.4 — Create lead_days and no_show features

df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

df["no_show"] = (
    df["status"] == "No-Show"
).astype(int)

print("Minimum lead days:", df["lead_days"].min())
print("Maximum lead days:", df["lead_days"].max())

print("\nTotal appointments:", len(df))
print("Total no-shows:", df["no_show"].sum())

print("\nNo-show rate (%):",
      round(df["no_show"].mean() * 100, 1))

print("\nDataset shape:", df.shape)


Minimum lead days: 1
Maximum lead days: 45

Total appointments: 990
Total no-shows: 218

No-show rate (%): 22.0

Dataset shape: (990, 16)


In [16]:

# Task 1.4 — NumPy array and column means

import numpy as np

# Create NumPy array using the four required columns
X = df[
    ["age", "distance_km", "fee_inr", "lead_days"]
].to_numpy()

# Calculate column means using NumPy
column_means = X.mean(axis=0)

print("NumPy array shape:", X.shape)

print("\nColumn means:")
for column, mean in zip(
    ["age", "distance_km", "fee_inr", "lead_days"],
    column_means
):
    print(f"{column}: {mean:.2f}")


NumPy array shape: (990, 4)

Column means:
age: 48.17
distance_km: 9.50
fee_inr: 635.71
lead_days: 11.82


In [17]:

# Task 1.4 — Appointments with lead_days > 14

long_lead_count = np.sum(df["lead_days"] > 14)

long_lead_percentage = (
    long_lead_count / len(df)
) * 100

print("Total appointments:", len(df))
print("Appointments with lead_days > 14:", long_lead_count)
print(f"Percentage with lead_days > 14: {long_lead_percentage:.1f}%")


Total appointments: 990
Appointments with lead_days > 14: 296
Percentage with lead_days > 14: 29.9%


In [18]:

# Task 1.4 — Distance z-score analysis

distance = df["distance_km"].to_numpy()

# Calculate mean and population standard deviation
distance_mean = np.mean(distance)
distance_std = np.std(distance)

# Calculate z-scores
distance_z_scores = (
    distance - distance_mean
) / distance_std

# Count appointments with absolute z-score > 2
outlier_z_count = np.sum(
    np.abs(distance_z_scores) > 2
)

print("Total appointments:", len(distance))
print("Appointments with |z| > 2:", outlier_z_count)


Total appointments: 990
Appointments with |z| > 2: 53


### Task 1.5 — Enrich clinics from a live API

In [20]:

import os

print("clinics.csv found:", os.path.exists("clinics.csv"))


clinics.csv found: True


In [21]:

# Task 1.5 — Step 1: Load and inspect clinics.csv

import pandas as pd

clinics_df = pd.read_csv("clinics.csv")

print("Dataset shape:", clinics_df.shape)

print("\nColumn names:")
print(clinics_df.columns.tolist())

print("\nClinic data:")
print(clinics_df.to_string(index=False))

print("\nMissing values:")
print(clinics_df.isnull().sum())


Dataset shape: (8, 6)

Column names:
['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']

Clinic data:
clinic_id              clinic_name    city   tier  latitude  longitude
      C01    Kothrud Family Clinic    Pune  Metro   18.5074    73.8077
      C02     Hadapsar Care Centre    Pune  Metro   18.5089    73.9260
      C03   Sitabuldi Health Point  Nagpur Tier-2   21.1458    79.0882
      C04     Malviya Nagar Clinic  Jaipur Tier-2   26.8553    75.8150
      C05    Vaishali Nagar Clinic  Jaipur Tier-2   26.9124    75.7873
      C06 Edappally Medical Centre   Kochi Tier-2   10.0261    76.3085
      C07       Vijay Nagar Clinic  Indore Tier-2   22.7533    75.8937
      C08   Gomti Nagar Care Point Lucknow Tier-2   26.8467    80.9462

Missing values:
clinic_id      0
clinic_name    0
city           0
tier           0
latitude       0
longitude      0
dtype: int64


In [22]:

# Task 1.5 — Fetch clinic temperatures from Open-Meteo

import requests

api_url = "https://api.open-meteo.com/v1/forecast"

temperatures = []

for _, clinic in clinics_df.iterrows():

    params = {
        "latitude": clinic["latitude"],
        "longitude": clinic["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    temp_c = None

    # Maximum two attempts per clinic
    for attempt in range(2):
        try:
            response = requests.get(
                api_url,
                params=params,
                timeout=10
            )

            if response.status_code == 200:
                data = response.json()

                hourly = data.get("hourly", {})
                values = hourly.get("temperature_2m")

                if values and values[0] is not None:
                    temp_c = values[0]
                    break
                else:
                    print(
                        f"{clinic['clinic_name']}: "
                        "Missing hourly temperature"
                    )
            else:
                print(
                    f"{clinic['clinic_name']}: "
                    f"HTTP {response.status_code}"
                )

        except requests.exceptions.RequestException as e:
            print(
                f"{clinic['clinic_name']}: "
                f"Request failed - {e}"
            )

        except ValueError as e:
            print(
                f"{clinic['clinic_name']}: "
                f"Invalid JSON - {e}"
            )

    temperatures.append(temp_c)

    print(
        f"{clinic['clinic_name']}: "
        f"temp_c = {temp_c}"
    )

clinics_df["temp_c"] = temperatures

print("\nFinal dataset shape:", clinics_df.shape)
print("Successful temperatures:",
      clinics_df["temp_c"].notna().sum())
print("Missing temperatures:",
      clinics_df["temp_c"].isna().sum())


Kothrud Family Clinic: temp_c = 16.7
Hadapsar Care Centre: temp_c = 18.0
Sitabuldi Health Point: temp_c = 20.1
Malviya Nagar Clinic: temp_c = 17.4
Vaishali Nagar Clinic: temp_c = 17.4
Edappally Medical Centre: temp_c = 24.9
Vijay Nagar Clinic: temp_c = 20.1
Gomti Nagar Care Point: temp_c = 16.4

Final dataset shape: (8, 7)
Successful temperatures: 8
Missing temperatures: 0


In [23]:

# Task 1.5 — Export enriched clinic data

clinics_df.to_csv(
    "clinics_enriched.csv",
    index=False
)

# Verify the saved CSV
saved_clinics = pd.read_csv("clinics_enriched.csv")

print("Saved file: clinics_enriched.csv")
print("Dataset shape:", saved_clinics.shape)

print("\nColumn names:")
print(saved_clinics.columns.tolist())

print("\nMissing temperatures:",
      saved_clinics["temp_c"].isnull().sum())

print("\nSaved clinic data:")
print(saved_clinics.to_string(index=False))


Saved file: clinics_enriched.csv
Dataset shape: (8, 7)

Column names:
['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude', 'temp_c']

Missing temperatures: 0

Saved clinic data:
clinic_id              clinic_name    city   tier  latitude  longitude  temp_c
      C01    Kothrud Family Clinic    Pune  Metro   18.5074    73.8077    16.7
      C02     Hadapsar Care Centre    Pune  Metro   18.5089    73.9260    18.0
      C03   Sitabuldi Health Point  Nagpur Tier-2   21.1458    79.0882    20.1
      C04     Malviya Nagar Clinic  Jaipur Tier-2   26.8553    75.8150    17.4
      C05    Vaishali Nagar Clinic  Jaipur Tier-2   26.9124    75.7873    17.4
      C06 Edappally Medical Centre   Kochi Tier-2   10.0261    76.3085    24.9
      C07       Vijay Nagar Clinic  Indore Tier-2   22.7533    75.8937    20.1
      C08   Gomti Nagar Care Point Lucknow Tier-2   26.8467    80.9462    16.4


### Task 1.6 — Export the clean table and write the SQL load script

In [24]:

# Task 1.6 — Step 1: Sort appointments and format dates

# Create a separate DataFrame for export
export_df = df.sort_values(
    by="appointment_id"
).copy()

# Format dates as YYYY-MM-DD
export_df["booked_on"] = (
    export_df["booked_on"].dt.strftime("%Y-%m-%d")
)

export_df["appt_date"] = (
    export_df["appt_date"].dt.strftime("%Y-%m-%d")
)

print("Dataset shape:", export_df.shape)

print("\nFirst 5 appointments:")
print(
    export_df[
        ["appointment_id", "booked_on", "appt_date"]
    ].head().to_string(index=False)
)

print("\nAppointment IDs sorted:",
      export_df["appointment_id"].is_monotonic_increasing)

print("\nDate column types:")
print(export_df[["booked_on", "appt_date"]].dtypes)


Dataset shape: (990, 16)

First 5 appointments:
appointment_id  booked_on  appt_date
         A0001 2025-01-24 2025-01-25
         A0002 2025-05-27 2025-06-06
         A0003 2025-03-24 2025-03-30
         A0004 2025-04-18 2025-04-22
         A0005 2024-11-23 2025-01-02

Appointment IDs sorted: True

Date column types:
booked_on    object
appt_date    object
dtype: object


In [25]:

# Task 1.6 — Step 2: Validate cleaned appointments

required_columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

# Arrange columns in the required order
export_df = export_df[required_columns].copy()

print("Dataset shape:", export_df.shape)

print("\nTotal appointment IDs:",
      export_df["appointment_id"].nunique())

print("Duplicate appointment IDs:",
      export_df["appointment_id"].duplicated().sum())

print("\nTotal missing values:",
      export_df.isnull().sum().sum())

print("\nColumns in correct order:",
      export_df.columns.tolist() == required_columns)

print("\nMissing values by column:")
print(export_df.isnull().sum())


Dataset shape: (990, 16)

Total appointment IDs: 990
Duplicate appointment IDs: 0

Total missing values: 0

Columns in correct order: True

Missing values by column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64


In [26]:

# Task 1.6 — Step 3: Export cleaned appointments

export_df.to_csv(
    "appointments_clean.csv",
    index=False
)

# Read the exported file to verify it
saved_appointments = pd.read_csv(
    "appointments_clean.csv"
)

print("Saved file: appointments_clean.csv")

print("Dataset shape:", saved_appointments.shape)

print("Unique appointment IDs:",
      saved_appointments["appointment_id"].nunique())

print("Missing values:",
      saved_appointments.isnull().sum().sum())

print("\nFirst 5 appointments:")
print(saved_appointments.head().to_string(index=False))


Saved file: appointments_clean.csv
Dataset shape: (990, 16)
Unique appointment IDs: 990
Missing values: 0

First 5 appointments:
appointment_id patient_id clinic_id  booked_on  appt_date  age gender  distance_km  fee_inr booking_channel   visit_type reminder_sent  prior_no_shows   status  lead_days  no_show
         A0001       P509       C05 2025-01-24 2025-01-25   60      F         18.0      850           Phone Consultation           Yes               1  No-Show          1        1
         A0002       P046       C07 2025-05-27 2025-06-06   39      M         12.5      350         Walk-In    Follow-up            No               2 Attended         10        0
         A0003       P009       C02 2025-03-24 2025-03-30   63      F          4.1      850         Walk-In Consultation            No               0 Attended          6        0
         A0004       P482       C01 2025-04-18 2025-04-22   52      F          7.5      750             App Consultation           Yes               0 